In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import warnings, os, re; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
from statsmodels.tsa.statespace.sarimax import SARIMAX

# paths
LEVELS_FILE  = "Code Outputs/Gap Interpolation Outputs/Unified_Interpolated_Levels.xlsx"
CLIMATE_FILE = "Code Outputs/Climate Data Extraction Outputs/Lake_Climate_Monthly.xlsx"
ORDERS_FILE  = "Code Outputs/Arima Forecast Outputs/FC_orders.csv"
DMI_FILE     = "Climate Indices/dmi.csv"
OUT_DIR      = "Code Outputs/Forecast Var Outputs"; os.makedirs(OUT_DIR, exist_ok=True)
def out(n): return os.path.join(OUT_DIR, n)

START, END  = "1995-06-01", "2025-12-01"
TEST_MONTHS = 60
HORIZONS    = [1, 3, 6, 12]
VAR_MAXLAG  = 4

def rmse(p, a): p, a = np.asarray(p, float), np.asarray(a, float); return np.sqrt(np.nanmean((p - a) ** 2))
def mae(p, a):  p, a = np.asarray(p, float), np.asarray(a, float); return np.nanmean(np.abs(p - a))

# load
lev = pd.read_excel(LEVELS_FILE); lev["Date"] = pd.to_datetime(lev["Date"])
L = (lev.pivot(index="Date", columns="Reservoir", values="Level_m")
        .sort_index().asfreq("MS").loc[START:END])
assert L.notna().all().all(), "NaNs in common window - check START date"
LAKES = list(L.columns); dL = L.diff()

clim = pd.read_excel(CLIMATE_FILE); clim["Date"] = pd.to_datetime(clim["Date"])
precip = clim.pivot(index="Date", columns="Reservoir", values="precip_mm").asfreq("MS")

SPLIT_CLIM = len(L) - TEST_MONTHS          # place after L and TEST_MONTHS exist

def deseason(s):                            # training-only climatology
    base = s.iloc[:SPLIT_CLIM]
    monthly = base.groupby(base.index.month).mean()
    return s - s.index.month.map(monthly).to_numpy()

precip_anom = precip.reindex(L.index).apply(deseason)   # reindex before deseasoning

def parse_index(path):
    if not os.path.exists(path): return None
    recs = []
    for line in open(path):
        t = re.split(r"[,\s]+", line.strip())
        if not t or t == [""]: continue
        m = re.match(r"(\d{4})[-/](\d{1,2})[-/](\d{1,2})$", t[0])
        if m and len(t) >= 2:
            try: recs.append((pd.Timestamp(int(m[1]), int(m[2]), 1), float(t[1])))
            except ValueError: pass
        elif len(t) == 13 and re.fullmatch(r"\d{4}", t[0]):
            try:
                for mo, v in enumerate([float(x) for x in t[1:]], 1):
                    recs.append((pd.Timestamp(int(t[0]), mo, 1), v))
            except ValueError: pass
    if not recs: return None
    s = pd.Series(dict(recs)).sort_index(); s[s < -90] = np.nan
    return s.asfreq("MS")

dmi = parse_index(DMI_FILE)
dmi = (dmi.reindex(L.index) if dmi is not None else pd.Series(0.0, index=L.index)).ffill().fillna(0.0)

orders = pd.read_csv(ORDERS_FILE).set_index("Lake")
def sarima_spec(lk): return eval(orders.loc[lk, "ARIMA_order"]), eval(orders.loc[lk, "SARIMA_seasonal"])

# ---- exogenous blocks ----------------------------------------------------
SEAS = pd.get_dummies(L.index.month, prefix="m", drop_first=True).astype(float); SEAS.index = L.index
CLIM = pd.concat([precip_anom.add_prefix("precip_"), dmi.rename("DMI")], axis=1).shift(1)
CLIM = CLIM.reindex(L.index).ffill().fillna(0.0)

N = len(L); split = N - TEST_MONTHS; targets = range(split, N)

# load the shared baseline instead of re-fitting SARIMA
BASE_PRED = "Code Outputs/Baseline Outputs/Baseline_predictions.csv"
bp = pd.read_csv(BASE_PRED, parse_dates=["Origin_Date", "Target_Date"])

assert set(bp["Lake"]) == set(LAKES), "baseline lakes differ from this script's"
idx_of = {d: i for i, d in enumerate(L.index)}

preds, loaded = {}, 0
for r in bp.itertuples(index=False):
    if r.Model not in ("RandomWalk", "SARIMA"):
        continue
    t = idx_of.get(r.Target_Date)
    if t is None:
        continue                      # baseline target outside this window
    preds[(r.Model, r.Lake, t, r.Horizon_m)] = r.Pred_m
    loaded += 1
print(f"  loaded {loaded} shared-baseline predictions (RandomWalk + SARIMA)")
assert loaded == len(LAKES) * 2 * (60 + 58 + 55 + 49), "baseline window mismatch"

# VAR / VARX
def var_rolling(exog_full, label):
    train0 = dL.iloc[1:split]
    ex0 = exog_full.iloc[1:split] if exog_full is not None else None
    p = max(1, int(VAR(train0, exog=ex0).select_order(VAR_MAXLAG).aic))
    print(f"  {label}: VAR lag order p={p}")
    for o in targets:
        tr = dL.iloc[1:o]
        ex_tr = exog_full.iloc[1:o] if exog_full is not None else None
        res = VAR(tr, exog=ex_tr).fit(p)
        H = min(max(HORIZONS), N - o)
        if exog_full is not None:
            fut_idx = L.index[o:o + H]
            ex_future = exog_full.loc[fut_idx].copy()
            # FUTURE-CLIMATE HANDLING (OFF-BY-ONE CORRECTED)
            # CLIM was built with .shift(1), so exog row at calendar index t holds
            # the climate OBSERVED in month t-1. Row `o` therefore holds month o-1,
            # which has fully elapsed at the forecast origin and is legitimately in
            # the information set. The previous version zeroed rows o..o+H-1 and so
            # discarded one month of genuinely observed climate. Only rows o+1
            # onwards are unobserved and must follow the policy.
            #   water balance / precip anomaly : 0 (climatological mean)
            #   DMI                            : held at its last OBSERVED value
            precip_cols = [c for c in exog_full.columns if c.startswith("precip_")]
            if precip_cols:
                ex_future.iloc[1:, [ex_future.columns.get_loc(c) for c in precip_cols]] = 0.0   #row 0 (= month o-1) kept
            if "DMI" in exog_full.columns:
                ex_future.iloc[1:, ex_future.columns.get_loc("DMI")] = exog_full["DMI"].iloc[o] #last OBSERVED DMI

            fc_d = res.forecast(tr.values[-p:], steps=H, exog_future=ex_future.values)
        else:
            fc_d = res.forecast(tr.values[-p:], steps=H)
        base = L.iloc[o - 1].values
        cum = base + np.cumsum(fc_d, axis=0)
        for h in HORIZONS:
            if h <= H:
                for j, lk in enumerate(LAKES):
                    preds[(label, lk, o + h - 1, h)] = cum[h - 1, j]

var_rolling(SEAS, "VAR")
var_rolling(pd.concat([SEAS, CLIM], axis=1), "VARX")

# score
MODELS = ["RandomWalk", "SARIMA", "VAR", "VARX"]
Lv = {lk: L[lk].values for lk in LAKES}
rows = []
for lk in LAKES:
    for model in MODELS:
        for h in HORIZONS:
            P, A = [], []
            for o in targets:
                tgt = o + h - 1
                if tgt >= N: continue
                key = (model, lk, tgt, h)
                if key in preds: P.append(preds[key]); A.append(Lv[lk][tgt])
            rows.append({"Lake": lk, "Model": model, "Horizon_m": h,
                         "RMSE_m": round(rmse(P, A), 4), "MAE_m": round(mae(P, A), 4)})
metrics = pd.DataFrame(rows)
sar = metrics[metrics.Model == "SARIMA"].set_index(["Lake", "Horizon_m"])["RMSE_m"]
metrics["skill_vs_SARIMA_%"] = metrics.apply(
    lambda r: round(100 * (sar[(r.Lake, r.Horizon_m)] - r.RMSE_m) / sar[(r.Lake, r.Horizon_m)], 1), axis=1)
metrics.to_csv(out("FC4_metrics.csv"), index=False)

print("\n=== MEAN skill vs SARIMA (%) by model x horizon (corrected VARX) ===")
print(metrics.groupby(["Model"]).apply(lambda d: d.groupby("Horizon_m")["skill_vs_SARIMA_%"].mean()).round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for ax, h in zip(axes, [1, 6]):
    sub = metrics[metrics.Horizon_m == h].pivot(index="Lake", columns="Model", values="RMSE_m")[MODELS]
    sub.plot(kind="bar", ax=ax); ax.set_title(f"RMSE at horizon {h} months", fontweight="bold")
    ax.set_ylabel("RMSE (m)"); ax.tick_params(axis="x", rotation=45)
plt.suptitle("Model comparison (corrected VARX): does cross-lake dependence beat SARIMA?", fontweight="bold")
plt.tight_layout(); plt.savefig(out("FC4_compare.png"), dpi=200); plt.close()
print("\nDone. Corrected FC4_metrics.csv + FC4_compare.png written to", OUT_DIR)

  loaded 3108 shared-baseline predictions (RandomWalk + SARIMA)
  VAR: VAR lag order p=3
  VARX: VAR lag order p=1

=== MEAN skill vs SARIMA (%) by model x horizon (corrected VARX) ===
Horizon_m     1     3     6    12
Model                            
RandomWalk -53.4 -65.8 -47.2 -0.8
SARIMA       0.0   0.0   0.0  0.0
VAR          0.5   2.7   3.3  0.3
VARX        11.7   6.0   4.1  3.0

Done. Corrected FC4_metrics.csv + FC4_compare.png written to Code Outputs/Forecast Var Outputs
